# Metodología del Notebook 3 - Análisis exploratorio de negocio

## Objetivo

El objetivo del tercer notebook fue realizar un análisis exploratorio de la base maestra para comprender el comportamiento general del ecommerce desde una perspectiva comercial, operativa, geográfica y de clientes. Esta fase permitió transformar la tabla analítica final en conocimiento útil de negocio a través de indicadores, agregaciones y visualizaciones.

## Procedimiento

La fase comenzó con la carga de la tabla maestra y la preparación de variables temporales derivadas de la fecha de compra, con el fin de analizar la evolución del negocio a lo largo del tiempo. A partir de estas variables se calcularon los KPIs generales del ecommerce, incluyendo volumen de pedidos, clientes únicos, ventas totales, ticket promedio, puntuación media de reseñas y porcentaje de pedidos tardíos.

Posteriormente, se desarrolló un análisis temporal a nivel mensual, que permitió estudiar la evolución de pedidos, ventas y ticket promedio. A continuación, se abordó la dimensión logística, restringiendo el análisis a los pedidos entregados para calcular tiempos reales de entrega, retrasos y proporción de pedidos tardíos.

En la dimensión de categorías, se elaboraron tablas agregadas para identificar las categorías más relevantes por volumen y ventas, así como aquellas con peor puntuación media o mayor proporción de retrasos. Del mismo modo, se construyó un análisis geográfico por estado, estudiando diferencias territoriales en número de pedidos, ventas, satisfacción y tardanzas.

Finalmente, se desarrolló una primera segmentación de clientes a partir de `customer_unique_id`, diferenciando entre clientes de compra única y clientes repetidores. Sobre esta base se analizaron también el gasto total acumulado, el ticket promedio y la contribución económica de los clientes de mayor valor.

## Justificación metodológica

Esta fase permitió estudiar el negocio antes de incorporar el componente textual de las reseñas. Desde un punto de vista metodológico, resultaba necesario comprender primero qué patrones existían en ventas, logística, categorías, territorio y clientes para, después, interpretar el análisis de sentimiento dentro de un contexto de negocio ya conocido.

## Resultado esperado

El resultado del notebook fue la identificación de patrones clave del ecommerce, tales como concentración del negocio en determinadas categorías y estados, diferencias geográficas en retrasos y satisfacción, y una base de clientes dominada por compradores de una sola vez, pero donde los clientes repetidores aportan mayor valor acumulado.

In [901]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
px.defaults.template = "plotly_white"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Carga de la tabla final que hemos construido en el notebook anterior:

In [902]:
master_df = pd.read_csv("../data/final/master_orders_dataset.csv")
master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,approval_days,carrier_days,delay_days,is_late,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,n_items,n_products,n_sellers,total_price,total_freight,payment_count,payment_value_total,max_installments,review_score,review_creation_date,review_answer_timestamp,review_comment_title,review_comment_message,main_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.00,0.00,2.00,-8.00,0.00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,"3,149.00",-23.58,-46.59,sao paulo,SP,1.00,1.00,1.00,29.99,8.72,3.00,38.71,1.00,4.00,2017-10-11 00:00:00,2017-10-12 03:43:48,NaN,"Não testei o produto ainda, mas ele veio corre...",housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.00,1.00,1.00,-6.00,0.00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,"47,813.00",-12.18,-44.66,barreiras,BA,1.00,1.00,1.00,118.70,22.76,1.00,141.46,1.00,4.00,2018-08-08 00:00:00,2018-08-08 18:37:50,Muito boa a loja,Muito bom o produto.,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.00,0.00,0.00,-18.00,0.00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,"75,265.00",-16.75,-48.51,vianopolis,GO,1.00,1.00,1.00,159.90,19.22,1.00,179.12,3.00,5.00,2018-08-18 00:00:00,2018-08-22 19:07:58,NaN,NaN,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.00,0.00,3.00,-13.00,0.00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,"59,296.00",-5.77,-35.27,sao goncalo do amarante,RN,1.00,1.00,1.00,45.00,27.20,1.00,72.20,1.00,5.00,2017-12-03 00:00:00,2017-12-05 19:21:58,NaN,O produto foi exatamente o que eu esperava e e...,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.00,0.00,0.00,-10.00,0.00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,"9,195.00",-23.68,-46.51,santo andre,SP,1.00,1.00,1.00,19.90,8.72,1.00,28.62,1.00,5.00,2018-02-17 00:00:00,2018-02-18 13:02:51,NaN,NaN,stationery


In [903]:
monthly = (master_df.assign(
    order_purchase_timestamp=pd.to_datetime(master_df["order_purchase_timestamp"]),
    purchase_year_month=lambda x: x["order_purchase_timestamp"].dt.to_period("M").astype(str),
    purchase_date=lambda x: x["order_purchase_timestamp"].dt.date
)
.groupby("purchase_year_month", as_index=False)
.agg(
    pedidos=("order_id", "count"),
    ventas=("payment_value_total", "sum"),
    ticket_promedio=("payment_value_total", "mean"),
    review_score_promedio=("review_score", "mean"),
    porcentaje_tardios=("is_late", "mean"),
    dias_activos=("purchase_date", "nunique")
))

monthly["ventas"] = monthly["ventas"].round(2)
monthly["ticket_promedio"] = monthly["ticket_promedio"].round(2)
monthly["review_score_promedio"] = monthly["review_score_promedio"].round(2)
monthly["porcentaje_tardios"] = (monthly["porcentaje_tardios"] * 100).round(2)
monthly["ym_dt"] = pd.to_datetime(monthly["purchase_year_month"])
monthly = monthly.sort_values("ym_dt").reset_index(drop=True)

monthly_business = monthly.query("'2017-01' <= purchase_year_month <= '2018-08'").copy()
monthly_business["ventas_ma3"] = monthly_business["ventas"].rolling(3).mean()
monthly_business["ticket_ma3"] = monthly_business["ticket_promedio"].rolling(3).mean()

- Verificamos la carga correcta del archivo

In [904]:
master_df.shape

(99441, 36)

In [905]:
monthly_summary[[
    "purchase_year_month", "pedidos", "ventas", "ticket_promedio",
    "fecha_min", "fecha_max", "dias_activos"
]]

,purchase_year_month,pedidos,ventas,ticket_promedio,fecha_min,fecha_max,dias_activos
0,2016-09,4,252.24,84.08,2016-09-04 21:15:19,2016-09-15 12:16:38,4
1,2016-10,324,"59,090.48",182.38,2016-10-02 22:07:52,2016-10-22 08:25:27,10
2,2016-12,1,19.62,19.62,2016-12-23 23:16:47,2016-12-23 23:16:47,1
3,2017-01,800,"138,488.04",173.11,2017-01-05 11:56:06,2017-01-31 23:37:58,27
4,2017-02,1780,"291,908.01",163.99,2017-02-01 00:04:17,2017-02-28 23:47:08,28
5,2017-03,2682,"449,863.60",167.73,2017-03-01 00:01:30,2017-03-31 23:54:45,31
6,2017-04,2404,"417,788.03",173.79,2017-04-01 00:54:10,2017-04-30 23:48:13,30
7,2017-05,3700,"592,918.82",160.25,2017-05-01 01:18:22,2017-05-31 22:59:52,31
8,2017-06,3245,"511,276.38",157.56,2017-06-01 00:05:38,2017-06-30 23:20:08,30
9,2017-07,4026,"592,382.92",147.14,2017-07-01 00:04:15,2017-07-31 23:55:27,31


#### creamos una vista general para revisar la cobertura mensual, ayudando a observar meses con pocos pedidos o dias activos para determinar los datos que aporten cierto valor al analisis 

In [906]:
monthly_business = monthly_summary[
    (monthly_summary["purchase_year_month_dt"] >= "2017-01-01") &
    (monthly_summary["purchase_year_month_dt"] <= "2018-08-31")
].copy()

monthly_business["ventas_ma3"] = monthly_business["ventas"].rolling(3).mean()
monthly_business["pedidos_ma3"] = monthly_business["pedidos"].rolling(3).mean()
monthly_business["ticket_ma3"] = monthly_business["ticket_promedio"].rolling(3).mean()

monthly_business.head()

,purchase_year_month,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios,fecha_min,fecha_max,dias_activos,purchase_year_month_dt,ventas_ma3,pedidos_ma3,ticket_ma3
3,2017-01,800,"138,488.04",173.11,4.06,2.93,2017-01-05 11:56:06,2017-01-31 23:37:58,27,2017-01-01,NaN,NaN,NaN
4,2017-02,1780,"291,908.01",163.99,4.02,2.96,2017-02-01 00:04:17,2017-02-28 23:47:08,28,2017-02-01,NaN,NaN,NaN
5,2017-03,2682,"449,863.60",167.73,4.07,4.56,2017-03-01 00:01:30,2017-03-31 23:54:45,31,2017-03-01,"293,419.88","1,754.00",168.28
6,2017-04,2404,"417,788.03",173.79,4.05,6.56,2017-04-01 00:54:10,2017-04-30 23:48:13,30,2017-04-01,"386,519.88","2,288.67",168.50
7,2017-05,3700,"592,918.82",160.25,4.14,2.99,2017-05-01 01:18:22,2017-05-31 22:59:52,31,2017-05-01,"486,856.82","2,928.67",167.26


## Preparamos las fechas utiles:
- Con la intención de convertir columnas de fecha a formato datetime.

In [907]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_cols:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col], errors="coerce")

master_df[date_cols].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
review_creation_date             datetime64[ns]
review_answer_timestamp          datetime64[ns]
dtype: object

## Creamos variables temporales para realización del analisis mensual y anual.

In [908]:
master_df["purchase_year"] = master_df["order_purchase_timestamp"].dt.year
master_df["purchase_month"] = master_df["order_purchase_timestamp"].dt.month
master_df["purchase_month_name"] = master_df["order_purchase_timestamp"].dt.month_name()
master_df["purchase_year_month"] = master_df["order_purchase_timestamp"].dt.to_period("M").astype(str)

master_df[[
    "order_purchase_timestamp",
    "purchase_year",
    "purchase_month",
    "purchase_month_name",
    "purchase_year_month"
]].head()

,order_purchase_timestamp,purchase_year,purchase_month,purchase_month_name,purchase_year_month
0,2017-10-02 10:56:33,2017,10,October,2017-10
1,2018-07-24 20:41:37,2018,7,July,2018-07
2,2018-08-08 08:38:49,2018,8,August,2018-08
3,2017-11-18 19:28:06,2017,11,November,2017-11
4,2018-02-13 21:18:39,2018,2,February,2018-02


## Interpretación de los KPIs para calcular los indicadores generales del ecommerce:

El cálculo de estos KPIs tiene como finalidad obtener una visión panorámica del desempeño del ecommerce a partir de la tabla maestra construida previamente. En concreto, se analizan variables relacionadas con el volumen de actividad comercial, la base de clientes, los ingresos generados, el ticket promedio, la satisfacción media reflejada en las reseñas y el porcentaje de pedidos tardíos.

La interpretación de estos indicadores constituye el punto de partida del análisis exploratorio, ya que permite identificar el nivel general de desempeño del negocio y establecer una base de comparación para las siguientes etapas del proyecto. Posteriormente, estos resultados se complementarán con análisis temporales, logísticos, geográficos y de comportamiento del cliente.

In [909]:
kpis = {
    "Total pedidos": int(len(master_df)),
    "Clientes únicos": int(master_df["customer_unique_id"].nunique()),
    "Ventas totales (R$)": round(master_df["payment_value_total"].sum(), 2),
    "Ticket promedio (R$)": round(master_df["payment_value_total"].mean(), 2),
    "Review score promedio": round(master_df["review_score"].mean(), 2),
    "% pedidos tardíos": round(master_df["is_late"].mean() * 100, 2)
}

kpis_df = pd.DataFrame(kpis.items(), columns=["KPI", "Valor"])
kpis_df

,KPI,Valor
0,Total pedidos,"99,441.00"
1,Clientes únicos,"96,096.00"
2,Ventas totales (R$),"16,008,872.12"
3,Ticket promedio (R$),160.99
4,Review score promedio,4.09
5,% pedidos tardíos,6.77


## Conclusiones del análisis mensual:

En esta sección se agrupan los pedidos por mes con el objetivo de observar la evolución temporal del negocio. A través de esta tabla se analizan tres métricas principales: el número de pedidos, las ventas totales y el ticket promedio por mes.

La finalidad de este análisis es identificar patrones de crecimiento, caídas puntuales o posibles comportamientos estacionales en la actividad del ecommerce. Esta lectura temporal permitirá entender si el negocio evoluciona de forma estable, si existen meses con mayor dinamismo comercial y si el valor medio de compra se mantiene constante o varía con el tiempo.

In [910]:
monthly_orders = (
    master_df
    .groupby("purchase_year_month", as_index=False)
    .agg(
        pedidos=("order_id", "count"),
        ventas=("payment_value_total", "sum"),
        ticket_promedio=("payment_value_total", "mean")
    )
)

monthly_orders

,purchase_year_month,pedidos,ventas,ticket_promedio
0,2016-09,4,252.24,84.08
1,2016-10,324,"59,090.48",182.38
2,2016-12,1,19.62,19.62
3,2017-01,800,"138,488.04",173.11
4,2017-02,1780,"291,908.01",163.99
5,2017-03,2682,"449,863.60",167.73
6,2017-04,2404,"417,788.03",173.79
7,2017-05,3700,"592,918.82",160.25
8,2017-06,3245,"511,276.38",157.56
9,2017-07,4026,"592,382.92",147.14


### Interpretación de la evolución mensual de pedidos y ventas

La evolución mensual muestra el crecimiento progresivo del ecommerce durante su etapa de consolidación, especialmente a partir de 2017. Para mejorar la interpretación, se combinan en una misma visualización el número de pedidos y las ventas, junto con una media móvil de tres meses que suaviza la volatilidad mensual.

La aparente caída al final de la serie no debe interpretarse automáticamente como una contracción real del negocio. Antes de extraer conclusiones, se verificó la cobertura temporal del dataset y se observó que los últimos meses presentan menor volumen de registros, lo que sugiere periodos incompletos o parcialmente capturados. Por ello, el análisis de tendencia se centra en el periodo más estable del negocio.

In [911]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=monthly_business["purchase_year_month"], y=monthly_business["pedidos"],
            name="Pedidos", opacity=.4, secondary_y=False)
fig.add_scatter(x=monthly_business["purchase_year_month"], y=monthly_business["ventas"],
                name="Ventas", mode="lines+markers", secondary_y=True)
fig.add_scatter(x=monthly_business["purchase_year_month"], y=monthly_business["ventas_ma3"],
                name="Tendencia ventas (3M)", mode="lines", line=dict(dash="dash"), secondary_y=True)

fig.update_layout(title="Evolución mensual de pedidos y ventas", title_x=.5, hovermode="x unified")
fig.update_yaxes(title_text="Pedidos", secondary_y=False)
fig.update_yaxes(title_text="Ventas (R$)", secondary_y=True)
fig.show()

## Resumen de la evolución de ventas:

En este gráfico se representa la evolución mensual de los pedidos y ventas totales del ecommerce. El objetivo es analizar cómo cambian los ingresos a lo largo del tiempo y comprobar si siguen una tendencia similar al volumen de pedidos o si presentan un comportamiento distinto.

Esta visualización permite detectar meses especialmente fuertes o débiles en términos de facturación, así como comparar la dinámica de crecimiento de los ingresos frente al crecimiento del número de pedidos.

### Nota metodológica sobre la caída final de la serie temporal

La caída observada al final del periodo no se interpreta directamente como un deterioro real del negocio. La revisión de la cobertura mensual muestra que algunos meses presentan menor volumen de registros y menor número de días activos, por lo que pueden reflejar periodos incompletos o parcialmente capturados. Para evitar conclusiones erróneas, la interpretación de tendencia se apoya en el tramo más estable del dataset.

In [912]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=monthly_business["purchase_year_month"], y=monthly_business["pedidos"],
            name="Pedidos", opacity=.35, secondary_y=False)
fig.add_scatter(x=monthly_business["purchase_year_month"], y=monthly_business["ticket_promedio"],
                name="Ticket promedio", mode="lines+markers", secondary_y=True)
fig.add_scatter(x=monthly_business["purchase_year_month"], y=monthly_business["ticket_ma3"],
                name="Tendencia ticket (3M)", mode="lines", line=dict(dash="dash"), secondary_y=True)

fig.update_layout(title="Ticket promedio y volumen de pedidos por mes", title_x=.5, hovermode="x unified")
fig.update_yaxes(title_text="Pedidos", secondary_y=False)
fig.update_yaxes(title_text="Ticket promedio (R$)", secondary_y=True)
fig.show()

## Conclusiones del ticket promedio mensual

Este gráfico permite analizar la evolución del valor medio de los pedidos a lo largo del tiempo. Su interpretación es importante para diferenciar si el crecimiento del negocio se debe únicamente al aumento del número de pedidos o también a un mayor gasto medio por compra.

El estudio del ticket promedio ayuda a entender mejor el comportamiento de compra de los clientes y la capacidad del ecommerce para generar más ingresos por transacción, en este grafico se han excluido datos que hacen referencia a los meses con muy pocos dias de actividad.

El ticket promedio se interpreta junto con el volumen de pedidos, ya que en meses con menor base transaccional esta métrica puede distorsionarse por compras atípicas. La visualización permite comprobar si los cambios en el valor medio por pedido responden a una mejora estructural del negocio o a un efecto de composición causado por menor volumen.

## Cruzamos pedidos + ticket promedio

#### Permite analizar si la caida viene por menos pedidos, menos ticket medio o incluso si ambas influyen en el mismo.

### Analisis de logística y tiempos de entrega:

In [913]:
delivered_df = master_df[master_df["order_status"] == "delivered"].copy()
delivered_df.shape

(96478, 40)

- 96,478 filas = lo que se traduce en 96,478 pedidos entregados
- 40 columnas = estamos trabajando con la tabla filtrada delivered_df, que además ya incluye columnas nuevas creadas en este notebook, como las variables temporales.

## Preparación del análisis logístico

En esta sección se filtran exclusivamente los pedidos con estado `delivered` para analizar de manera precisa el comportamiento logístico del ecommerce. Esta decisión es necesaria porque las métricas de tiempo de entrega, retraso y cumplimiento solo tienen sentido en aquellos pedidos que efectivamente llegaron al cliente.

A partir de este subconjunto se podrán calcular indicadores relacionados con la eficiencia operativa y la experiencia de entrega, que posteriormente serán relevantes para interpretar la satisfacción del cliente y conectar este análisis con las reseñas.

### Definición de KPI's logísticos:

In [914]:
logistics_kpis = {
    "Tiempo entrega promedio (días)": round(delivered_df["delivery_days"].mean(), 2),
    "Retraso promedio (días)": round(delivered_df["delay_days"].mean(), 2),
    "% pedidos tardíos entregados": round(delivered_df["is_late"].mean() * 100, 2)
}

logistics_kpis_df = pd.DataFrame(logistics_kpis.items(), columns=["KPI", "Valor"])
logistics_kpis_df

,KPI,Valor
0,Tiempo entrega promedio (días),12.09
1,Retraso promedio (días),-11.88
2,% pedidos tardíos entregados,6.77


## Descripción de días de entrega:

In [915]:
delivery_summary = (
    delivered_df["delivery_days"]
    .describe(percentiles=[0.25, 0.50, 0.75])
    .rename({
        "count": "Número de pedidos entregados",
        "mean": "Media",
        "std": "Desviación estándar",
        "min": "Mínimo",
        "25%": "Percentil 25",
        "50%": "Mediana",
        "75%": "Percentil 75",
        "max": "Máximo"
    })
    .to_frame(name="Valor")
)

delivery_summary["Valor"] = delivery_summary["Valor"].round(2)
delivery_summary

,Valor
Número de pedidos entregados,"96,470.00"
Media,12.09
Desviación estándar,9.55
Mínimo,0.00
Percentil 25,6.00
Mediana,10.00
Percentil 75,15.00
Máximo,209.00


### Histogramas de dias de entrega:

In [916]:
d = delivered_df["delivery_days"].dropna()
p99 = d.quantile(0.99)
p90 = d.quantile(0.90)
med = d.median()
mean = d.mean()

fig = px.histogram(
    x=d[d <= p99],
    nbins=24,
    title="Distribución de días de entrega (hasta percentil 99)",
    opacity=0.9
)

fig.add_vline(x=med,  line_dash="dash", line_color="orange")
fig.add_vline(x=mean, line_dash="dot",  line_color="red")
fig.add_vline(x=p90,  line_dash="dash", line_color="green")

fig.add_annotation(x=med,  y=0.98, yref="paper", text=f"Mediana: {med:.1f} días", showarrow=False, xanchor="left")
fig.add_annotation(x=mean, y=0.90, yref="paper", text=f"Media: {mean:.1f} días", showarrow=False, xanchor="left")
fig.add_annotation(x=p90,  y=0.82, yref="paper", text=f"P90: {p90:.1f} días", showarrow=False, xanchor="left")

fig.update_layout(
    xaxis_title="Días de entrega",
    yaxis_title="Frecuencia",
    title_x=0.5,
    bargap=0.03
)

fig.show()

### Interpretación de la distribución de días de entrega

La distribución de los días de entrega muestra que la mayoría de los pedidos se concentra en rangos relativamente bajos, mientras que existe una cola hacia la derecha que refleja la presencia de entregas considerablemente más tardías. Para mejorar la legibilidad y evitar que unos pocos casos extremos distorsionen el gráfico, la visualización se acota hasta el percentil 99.

La mediana representa el comportamiento típico del proceso logístico, la media recoge el efecto de los retrasos más largos y el percentil 90 indica el umbral a partir del cual se concentran las entregas más lentas. Desde un punto de vista de negocio, esta lectura permite evaluar no solo la eficiencia media del sistema, sino también el riesgo operativo asociado a una minoría de pedidos con plazos claramente superiores a lo habitual.

### Porcentajes de pedidos tardios:

In [917]:
late_summary = (
    delivered_df["is_late"]
    .value_counts(normalize=True)
    .mul(100)
    .rename(index={0.0: "A tiempo", 1.0: "Tardío"})
    .reset_index()
)

late_summary.columns = ["Estado de entrega", "Porcentaje"]
late_summary["Porcentaje"] = late_summary["Porcentaje"].round(2)

late_summary

,Estado de entrega,Porcentaje
0,A tiempo,93.23
1,Tardío,6.77


## Resumen del porcentaje de pedidos tardíos

El análisis muestra que la mayoría de los pedidos entregados llegaron dentro del plazo previsto, mientras que una proporción menor presentó retraso. En concreto, el 93.23% de los pedidos fueron entregados a tiempo, frente a un 6.77% de pedidos tardíos.

Aunque el porcentaje de retrasos no representa la mayoría de la operación, sigue siendo una métrica relevante desde el punto de vista del negocio, ya que la demora en la entrega puede afectar negativamente la experiencia del cliente. Por ello, esta variable será especialmente importante en los siguientes análisis, donde se estudiará su relación con las valoraciones y con el sentimiento expresado en las reseñas.

## Analisis por categorias:

In [918]:
master_df["main_category"] = master_df["main_category"].fillna("Unknown")

category_summary = (
    master_df
    .groupby("main_category", as_index=False)
    .agg(
        pedidos=("order_id", "count"),
        ventas=("payment_value_total", "sum"),
        ticket_promedio=("payment_value_total", "mean"),
        review_score_promedio=("review_score", "mean"),
        porcentaje_tardios=("is_late", "mean")
    )
)

category_summary["ventas"] = category_summary["ventas"].round(2)
category_summary["ticket_promedio"] = category_summary["ticket_promedio"].round(2)
category_summary["review_score_promedio"] = category_summary["review_score_promedio"].round(2)
category_summary["porcentaje_tardios"] = (category_summary["porcentaje_tardios"] * 100).round(2)

category_summary.head()

,main_category,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,Unknown,2185,"370,173.54",169.42,3.17,7.11
1,agro_industry_and_commerce,182,"78,379.19",430.65,4.02,3.95
2,air_conditioning,253,"62,045.33",245.24,4.03,4.07
3,art,201,"28,408.98",141.34,4.04,6.19
4,arts_and_craftmanship,23,"2,241.54",97.46,4.17,4.35


### Resumen por categoría

En esta tabla se consolida el desempeño de cada categoría principal a partir de cinco métricas clave: volumen de pedidos, ventas totales, ticket promedio, valoración media de los clientes y porcentaje de pedidos tardíos. Esta estructura permite pasar de una lectura puramente descriptiva a una interpretación más estratégica del negocio.

Lo primero que se observa es que no todas las categorías aportan valor de la misma forma. Algunas destacan por volumen de pedidos, otras por capacidad de generar ingresos y otras por un ticket promedio más alto. Esto significa que el peso de una categoría no debe medirse solo por cuántos pedidos concentra, sino también por el valor económico que aporta y por la calidad de experiencia que ofrece al cliente.

También resulta relevante la presencia de la categoría `Unknown`, que agrupa productos sin clasificación clara. Aunque concentra un volumen y unas ventas significativas, su `review_score_promedio` es inferior al de muchas categorías identificadas. Desde el punto de vista analítico, esto sugiere que conviene tratar esta categoría con cautela en las conclusiones finales o incluso excluirla de algunos rankings estratégicos, ya que no representa una línea de negocio interpretable desde el punto de vista comercial.

En términos operativos, el `porcentaje_tardios` permite identificar si determinadas categorías combinan peso comercial con mayor fricción logística. Esto es importante porque una categoría puede ser rentable en ventas, pero presentar un mayor riesgo de deterioro en la experiencia del cliente si concentra más retrasos.

En conjunto, esta tabla sirve como base para priorizar análisis posteriores. Permite detectar qué categorías son estratégicas para el negocio, cuáles tienen mayor potencial económico, cuáles podrían estar afectadas por problemas operativos y cuáles requieren una revisión más detallada en términos de satisfacción y rentabilidad.

In [919]:
top10_sales_categories = (
    category_summary
    .sort_values("ventas", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top10_sales_categories

,main_category,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,health_beauty,8810,"1,443,182.68",163.83,4.18,7.54
1,watches_gifts,5584,"1,299,139.04",232.65,4.07,7.45
2,bed_bath_table,9385,"1,257,907.93",134.03,3.97,7.46
3,sports_leisure,7668,"1,154,092.27",150.51,4.18,6.59
4,computers_accessories,6679,"1,063,925.67",159.29,4.03,6.40
5,furniture_decor,6350,"907,045.66",142.84,4.02,7.18
6,housewares,5812,"775,139.40",133.37,4.15,5.43
7,cool_stuff,3589,"716,406.30",199.61,4.19,5.94
8,auto,3891,"687,648.92",176.73,4.09,7.31
9,garden_tools,3479,"583,237.30",167.65,4.15,6.60


### Graficos TOP 10 de categorías por ventas:

In [920]:
sales_plot = top10_sales_categories.sort_values("ventas").copy()

fig = px.bar(
    sales_plot, x="ventas", y="main_category", orientation="h",
    color="review_score_promedio",
    title="Top 10 categorías por ventas",
    hover_data={"ventas":":,.0f","pedidos":":,.0f","ticket_promedio":":.2f","porcentaje_tardios":":.2f","review_score_promedio":":.2f","main_category":False}
)
fig.update_layout(xaxis_title="Ventas (R$)", yaxis_title="Categoría", coloraxis_colorbar_title="Review score", title_x=.5)
fig.show()

## Conclusiones del top de categorías por ventas

La visualización permite identificar con mayor claridad qué categorías lideran el negocio en términos de ventas, incorporando además métricas complementarias como el número de pedidos, el ticket promedio, el porcentaje de retrasos y la puntuación media de reseñas. Esto facilita una lectura más completa, ya que no solo muestra qué categorías generan más ingresos, sino también si ese peso comercial viene acompañado de una experiencia de cliente sólida o de señales de fricción operativa.

## Categorias por numeros de pedidos:

In [921]:
top10_order_categories = (
    category_summary
    .sort_values("pedidos", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top10_order_categories

,main_category,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,bed_bath_table,9385,"1,257,907.93",134.03,3.97,7.46
1,health_beauty,8810,"1,443,182.68",163.83,4.18,7.54
2,sports_leisure,7668,"1,154,092.27",150.51,4.18,6.59
3,computers_accessories,6679,"1,063,925.67",159.29,4.03,6.40
4,furniture_decor,6350,"907,045.66",142.84,4.02,7.18
5,housewares,5812,"775,139.40",133.37,4.15,5.43
6,watches_gifts,5584,"1,299,139.04",232.65,4.07,7.45
7,telephony,4177,"393,945.50",94.31,4.01,7.15
8,auto,3891,"687,648.92",176.73,4.09,7.31
9,toys,3841,"556,903.78",144.99,4.20,6.41


In [922]:
top10_order_categories = (
    category_summary
    .sort_values("pedidos", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

fig = px.bar(
    top10_order_categories.sort_values("pedidos"),
    x="pedidos",
    y="main_category",
    orientation="h",
    color="porcentaje_tardios",
    title="Top 10 categorías por número de pedidos",
    hover_data=["ventas", "ticket_promedio", "review_score_promedio"]
)

fig.update_layout(
    xaxis_title="Pedidos",
    yaxis_title="Categoría"
)

fig.show()

In [923]:
category_bubble = category_summary[
    (category_summary["main_category"] != "Unknown") &
    (category_summary["pedidos"] >= 300)
].copy()

fig = px.scatter(
    category_bubble,
    x="review_score_promedio",
    y="porcentaje_tardios",
    size="ventas",
    color="main_category",
    hover_name="main_category",
    title="Relación entre satisfacción, retrasos y ventas por categoría",
    labels={
        "review_score_promedio": "Review score promedio",
        "porcentaje_tardios": "% pedidos tardíos"
    }
)

fig.update_layout(showlegend=False)
fig.show()

In [924]:
top_categories_heat = (
    category_summary
    .sort_values("ventas", ascending=False)
    .head(10)
    .set_index("main_category")[["review_score_promedio", "porcentaje_tardios", "ticket_promedio"]]
)

fig = px.imshow(
    top_categories_heat,
    text_auto=True,
    aspect="auto",
    title="Comparativa de métricas clave en las principales categorías"
)

fig.show()

### Comparativa de métricas clave en las principales categorías

La matriz permite comparar de forma simultánea tres dimensiones relevantes del negocio en las categorías con mayor peso comercial: satisfacción del cliente (`review_score_promedio`), fricción logística (`porcentaje_tardios`) y valor medio por transacción (`ticket_promedio`).

Desde una perspectiva de negocio, se observa que las categorías líderes no presentan exactamente el mismo perfil de desempeño. Por ejemplo, `watches_gifts` destaca por tener el ticket promedio más alto del grupo, lo que sugiere una mayor capacidad de generar ingresos por pedido, mientras que categorías como `bed_bath_table` o `health_beauty` combinan gran relevancia comercial con porcentajes de tardíos relativamente elevados.

También se aprecia que las diferencias en `review_score_promedio` son más estrechas que las diferencias en ticket y tardíos. Esto indica que, entre las categorías principales, la satisfacción media se mantiene en niveles relativamente similares, pero la experiencia operativa y el valor económico por pedido sí varían de manera más clara.

Este tipo de visualización es útil porque permite evitar lecturas aisladas. Una categoría puede vender mucho, pero presentar mayor fricción logística; otra puede tener mejor eficiencia, pero menor ticket promedio. Por tanto, la priorización estratégica no debería hacerse solo por ventas, sino considerando conjuntamente rentabilidad, calidad de experiencia y riesgo operativo.

### Top 10 de categorias por número de pedidos:

In [925]:
fig = px.scatter(
    top10_order_categories,
    x="pedidos", y="ventas", size="ticket_promedio", color="porcentaje_tardios",
    text="main_category",
    title="Top categorías: volumen, ventas, ticket y retrasos",
    hover_data={"pedidos":":,.0f","ventas":":,.0f","ticket_promedio":":.2f","review_score_promedio":":.2f","porcentaje_tardios":":.2f"}
)
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="Pedidos", yaxis_title="Ventas (R$)", coloraxis_colorbar_title="% tardíos", title_x=.5)
fig.show()

## Interpretación de las categorías con mayor volumen de pedidos

En esta visualización se analizan las categorías con mayor número de pedidos, independientemente del valor monetario asociado. Este enfoque permite distinguir entre categorías que destacan por volumen de compra y aquellas que generan mayor facturación.

La comparación entre pedidos y ventas resulta útil para detectar categorías de alta rotación frente a categorías de mayor valor unitario, lo que aporta una lectura más completa del comportamiento comercial del ecommerce.

## Categorias con peor review score:

In [926]:
category_quality = category_summary[
    (category_summary["pedidos"] >= 300) &
    (category_summary["main_category"] != "Unknown")
].copy()

worst_review_categories = (
    category_quality
    .sort_values("review_score_promedio", ascending=True)
    .head(10)
    .reset_index(drop=True)
)

worst_review_categories

,main_category,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,office_furniture,1263,"341,237.40",270.18,3.63,8.12
1,audio,350,"58,090.38",165.97,3.83,11.78
2,home_confort,351,"61,859.96",176.24,3.92,10.12
3,bed_bath_table,9385,"1,257,907.93",134.03,3.97,7.46
4,telephony,4177,"393,945.50",94.31,4.01,7.15
5,furniture_decor,6350,"907,045.66",142.84,4.02,7.18
6,computers_accessories,6679,"1,063,925.67",159.29,4.03,6.40
7,baby,2870,"487,297.73",169.79,4.05,8.09
8,furniture_living_room,411,"85,703.90",208.53,4.06,7.69
9,home_construction,467,"93,570.08",200.36,4.07,6.30


## Conclusión de las categorías con peor review score

Tras aplicar un filtro mínimo de 300 pedidos y excluir la categoría técnica `Unknown`, los resultados muestran con mayor claridad qué categorías presentan una experiencia de cliente más débil. En este contexto, `office_furniture` destaca como la categoría con peor puntuación media de reseñas, combinando además un porcentaje elevado de pedidos tardíos, lo que sugiere una posible fricción tanto comercial como operativa.

También sobresalen `audio` y `home_confort`, que muestran una combinación de baja satisfacción y altos niveles de retraso. Estos casos refuerzan la hipótesis de que, en determinadas categorías, el desempeño logístico puede estar influyendo negativamente en la percepción del cliente. Por otro lado, categorías como `bed_bath_table` resultan especialmente relevantes, ya que además de presentar señales de fricción, concentran un volumen muy alto de pedidos y ventas, lo que las convierte en una prioridad para el negocio.

### Categorías con mayor porcentaje de tardíos:

In [927]:
worst_late_categories = (
    category_quality
    .sort_values("porcentaje_tardios", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

worst_late_categories

,main_category,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,audio,350,"58,090.38",165.97,3.83,11.78
1,home_confort,351,"61,859.96",176.24,3.92,10.12
2,office_furniture,1263,"341,237.40",270.18,3.63,8.12
3,baby,2870,"487,297.73",169.79,4.05,8.09
4,furniture_living_room,411,"85,703.90",208.53,4.06,7.69
5,electronics,2542,"208,546.16",82.04,4.10,7.65
6,musical_instruments,623,"209,983.17",337.05,4.19,7.59
7,health_beauty,8810,"1,443,182.68",163.83,4.18,7.54
8,bed_bath_table,9385,"1,257,907.93",134.03,3.97,7.46
9,watches_gifts,5584,"1,299,139.04",232.65,4.07,7.45


### Interpretación de las categorías con mayor porcentaje de pedidos tardíos

Esta tabla permite identificar qué categorías presentan una mayor proporción de retrasos dentro del conjunto analizado, incorporando además variables de contexto como volumen de pedidos, ventas, ticket promedio y valoración media de los clientes. Esta combinación es importante porque evita interpretar el retraso de forma aislada.

En primer lugar, se observa que categorías como `audio` y `home_confort` encabezan el ranking de tardíos, con porcentajes superiores al resto del grupo. Sin embargo, su peso comercial es muy distinto al de otras categorías más relevantes en volumen, como `health_beauty`, `bed_bath_table` o `watches_gifts`, que también aparecen en el listado aunque con una escala de negocio mucho mayor.

Desde una perspectiva de negocio, esto significa que no todas las categorías con retrasos elevados deben priorizarse igual. Algunas presentan una tasa alta, pero sobre una base relativamente pequeña de pedidos, mientras que otras combinan una proporción de tardíos relevante con un volumen y unas ventas mucho más altos. Estas últimas son especialmente sensibles, porque cualquier problema logístico puede tener un impacto más amplio tanto en ingresos como en experiencia del cliente.

También resulta importante observar la relación con `review_score_promedio`. En varias categorías con más tardíos, la valoración media tiende a situarse en niveles relativamente más bajos, lo que sugiere que la logística puede estar afectando negativamente la percepción del cliente. Esto refuerza la idea de que los retrasos no son solo un problema operativo, sino también comercial y reputacional.

En conjunto, esta tabla ayuda a priorizar acciones de mejora logística en aquellas categorías donde coinciden tres factores: una tasa relevante de retraso, un peso comercial significativo y una posible afectación en la satisfacción del cliente.

# Analisis Geografico

In [928]:
state_summary = (
    master_df
    .groupby("geolocation_state", as_index=False)
    .agg(
        pedidos=("order_id", "count"),
        ventas=("payment_value_total", "sum"),
        ticket_promedio=("payment_value_total", "mean"),
        review_score_promedio=("review_score", "mean"),
        porcentaje_tardios=("is_late", "mean")
    )
)

state_summary["ventas"] = state_summary["ventas"].round(2)
state_summary["ticket_promedio"] = state_summary["ticket_promedio"].round(2)
state_summary["review_score_promedio"] = state_summary["review_score_promedio"].round(2)
state_summary["porcentaje_tardios"] = (state_summary["porcentaje_tardios"] * 100).round(2)

state_summary.head()

,geolocation_state,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,AC,81,"19,680.62",242.97,4.05,3.75
1,AL,412,"96,780.44",234.90,3.76,21.46
2,AM,148,"27,966.93",188.97,4.21,2.76
3,AP,68,"16,262.80",239.16,4.19,2.99
4,BA,3371,"615,203.62",182.50,3.86,12.10


In [929]:
brazil_state_coords = pd.DataFrame({
    "geolocation_state": ["AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT","PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO"],
    "lat": [-8.77,-9.71,-3.47,1.41,-12.97,-3.72,-15.78,-20.32,-16.64,-2.55,-19.92,-20.45,-15.60,-1.45,-7.12,-8.05,-5.09,-25.43,-22.91,-5.79,-8.76,2.82,-30.03,-27.59,-10.90,-23.55,-10.25],
    "lon": [-70.55,-35.73,-65.10,-51.77,-38.50,-38.54,-47.93,-40.34,-49.31,-44.30,-43.94,-54.62,-56.10,-48.50,-34.86,-34.90,-42.80,-49.27,-43.17,-35.21,-63.90,-60.67,-51.23,-48.55,-37.07,-46.63,-48.33]
})

In [930]:
state_map_df = state_summary.merge(
    brazil_state_coords,
    on="geolocation_state",
    how="left"
)

state_map_df.head()

,geolocation_state,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios,lat,lon
0,AC,81,"19,680.62",242.97,4.05,3.75,-8.77,-70.55
1,AL,412,"96,780.44",234.90,3.76,21.46,-9.71,-35.73
2,AM,148,"27,966.93",188.97,4.21,2.76,-3.47,-65.10
3,AP,68,"16,262.80",239.16,4.19,2.99,1.41,-51.77
4,BA,3371,"615,203.62",182.50,3.86,12.10,-12.97,-38.50


## Resumen geográfico por estado

En esta sección se construye una tabla agregada por estado con el objetivo de analizar cómo se distribuye el negocio geográficamente dentro del ecommerce. Para cada estado se calculan el número de pedidos, las ventas totales, el ticket promedio, la puntuación media de las reseñas y el porcentaje de pedidos tardíos.

Este análisis permite identificar qué estados concentran mayor actividad comercial y si existen diferencias territoriales relevantes en la experiencia del cliente y en el comportamiento logístico.

## TOP 10 de estados por pedido:

In [931]:
top_states_orders = (
    state_summary
    .sort_values("pedidos", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_states_orders

,geolocation_state,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,SP,41731,"5,996,050.41",143.69,4.17,4.49
1,RJ,12839,"2,142,349.25",166.86,3.88,12.12
2,MG,11624,"1,871,072.52",160.97,4.14,4.57
3,RS,5462,"890,584.82",163.05,4.13,6.07
4,PR,5034,"810,280.15",160.96,4.18,4.01
5,SC,3637,"623,086.43",171.32,4.07,8.20
6,BA,3371,"615,203.62",182.50,3.86,12.10
7,ES,2027,"325,490.67",160.58,4.04,10.61
8,GO,2011,"349,377.13",173.73,4.04,6.57
9,DF,1969,"326,095.81",165.61,4.07,5.74


## Interpretación de los estados con mayor número de pedidos

El ranking por número de pedidos muestra que la actividad comercial del ecommerce está fuertemente concentrada en unos pocos estados, con São Paulo (`SP`) liderando de forma muy destacada respecto al resto. Esto confirma que el volumen de demanda no se distribuye de manera homogénea y que una parte importante del negocio depende de mercados geográficos concretos.

Además, el análisis revela que no todos los estados con mayor volumen presentan el mismo nivel de desempeño operativo. Mientras territorios como `SP`, `MG` y `PR` combinan una base amplia de pedidos con indicadores relativamente sólidos de satisfacción y cumplimiento logístico, otros estados relevantes como `RJ` y `BA` muestran porcentajes de pedidos tardíos significativamente más altos y puntuaciones medias de reseñas más bajas. Esto sugiere que, dentro de los mercados más importantes, existen diferencias territoriales claras en la calidad de la experiencia del cliente.

In [932]:
fig = px.scatter_geo(
    state_map_df,
    lat="lat",
    lon="lon",
    size="ventas",
    color="ventas",
    hover_name="geolocation_state",
    custom_data=["ventas", "pedidos", "review_score_promedio", "porcentaje_tardios"],
    scope="south america",
    title="Distribución geográfica de ventas por estado",
    size_max=28,
    height=560,
    width=900
)

fig.update_geos(
    center={"lat": -14.2, "lon": -51.9},
    projection_scale=4.0,
    showcountries=True,
    showland=True
)

fig.update_layout(
    title_x=0.5,
    coloraxis_colorbar_title="Ventas",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.update_traces(
    hovertemplate=
    "<b>%{hovertext}</b><br>" +
    "Ventas: R$ %{customdata[0]:,.0f}<br>" +
    "Pedidos: %{customdata[1]:,.0f}<br>" +
    "Review score: %{customdata[2]:.2f}<br>" +
    "% pedidos tardíos: %{customdata[3]:.2f}%<extra></extra>"
)

fig.show()

- En esta tabla vemos como estados grandes y con evidente mayor indice poblacional son los que dominan el numero de pedidos, entre ellos destacan, Sao Paulo, Rio de Janeiro y Mina Gerais

### Resultado de la distribución geográfica de ventas

El mapa permite identificar los estados que concentran el mayor volumen de ventas dentro del ecommerce, mostrando de manera visual el peso comercial de cada territorio. El tamaño de la burbuja refleja el nivel de facturación, mientras que el detalle emergente incorpora información adicional sobre pedidos, satisfacción media y porcentaje de retrasos.

Desde un enfoque de negocio, esta visualización ayuda a detectar los mercados más relevantes para la compañía y permite contextualizar si ese liderazgo comercial está acompañado por una experiencia de cliente y una operativa logística consistentes. Esto facilita priorizar estados clave tanto para decisiones comerciales como para mejoras operativas.

In [933]:
states_plot = top_states_orders.sort_values("pedidos").copy()

fig = px.bar(
    states_plot, x="pedidos", y="geolocation_state", orientation="h",
    color="porcentaje_tardios", text="pedidos",
    title="Top 10 estados por número de pedidos",
    hover_data={"ventas":":,.0f","ticket_promedio":":.2f","review_score_promedio":":.2f","porcentaje_tardios":":.2f","geolocation_state":False}
)
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(xaxis_title="Pedidos", yaxis_title="Estado", coloraxis_colorbar_title="% tardíos", title_x=.5)
fig.show()

## Resolución de los estados con mayor volumen de pedidos

Esta visualización permite identificar los estados que concentran la mayor parte de la demanda del ecommerce en términos de número de pedidos. Su lectura es importante para detectar en qué zonas geográficas se encuentra el mayor volumen de actividad comercial.

El análisis por número de pedidos aporta una primera aproximación al peso relativo de cada estado dentro del negocio y servirá como base para comparar posteriormente la distribución territorial de las ventas, la satisfacción y los retrasos logísticos.

Los estados con mayor volumen concentran una parte muy relevante de la actividad comercial del ecommerce. Sin embargo, el color permite añadir una segunda lectura: no todos los mercados grandes operan con la misma calidad logística. Esta visualización ayuda a distinguir territorios con alto peso comercial y, al mismo tiempo, con más tensión operativa.

# TOP 10 Estados por Ventas:

In [934]:
top_states_sales = (
    state_summary
    .sort_values("ventas", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_states_sales

,geolocation_state,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,SP,41731,"5,996,050.41",143.69,4.17,4.49
1,RJ,12839,"2,142,349.25",166.86,3.88,12.12
2,MG,11624,"1,871,072.52",160.97,4.14,4.57
3,RS,5462,"890,584.82",163.05,4.13,6.07
4,PR,5034,"810,280.15",160.96,4.18,4.01
5,SC,3637,"623,086.43",171.32,4.07,8.20
6,BA,3371,"615,203.62",182.50,3.86,12.10
7,GO,2011,"349,377.13",173.73,4.04,6.57
8,DF,1969,"326,095.81",165.61,4.07,5.74
9,ES,2027,"325,490.67",160.58,4.04,10.61


## Interpretación de los estados con mayor volumen de ventas

Los resultados muestran que São Paulo (`SP`) concentra con diferencia la mayor parte de las ventas del ecommerce, confirmando su papel como principal mercado del negocio. Además de liderar en volumen económico, `SP` mantiene una puntuación media de reseñas elevada y un porcentaje de pedidos tardíos relativamente bajo, lo que sugiere un desempeño sólido tanto a nivel comercial como operativo.

También destacan estados como Río de Janeiro (`RJ`), Minas Gerais (`MG`) y Bahía (`BA`), aunque con comportamientos distintos. Mientras `MG` combina peso comercial con indicadores logísticos y de satisfacción estables, `RJ` y `BA` presentan porcentajes de retraso significativamente más altos y puntuaciones medias de reseñas más bajas, lo que apunta a una experiencia de cliente más débil en territorios comercialmente importantes.

In [935]:
top_states_sales = (
    state_summary
    .sort_values("ventas", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

fig = px.bar(
    top_states_sales.sort_values("ventas"),
    x="ventas",
    y="geolocation_state",
    orientation="h",
    color="porcentaje_tardios",
    title="Top 10 estados por ventas",
    hover_data=["pedidos", "ticket_promedio", "review_score_promedio"]
)

fig.update_layout(
    xaxis_title="Ventas",
    yaxis_title="Estado"
)

fig.show()

## Análisis de los estados con mayor volumen de ventas

En este gráfico se representan los estados con mayor contribución a las ventas totales del ecommerce. Esta visualización permite diferenciar entre territorios que destacan por volumen de pedidos y aquellos que generan un mayor valor económico.

La comparación entre pedidos y ventas ayuda a detectar posibles diferencias en el ticket promedio entre estados, así como regiones especialmente relevantes desde el punto de vista comercial.

# TOP 10 Estados con mayor concentracion de reviews más bajos:

In [936]:
state_quality = state_summary[state_summary["pedidos"] >= 500].copy()

worst_states_reviews = (
    state_quality
    .sort_values("review_score_promedio", ascending=True)
    .head(10)
    .reset_index(drop=True)
)

worst_states_reviews

,geolocation_state,pedidos,ventas,ticket_promedio,review_score_promedio,porcentaje_tardios
0,MA,743,"151,529.48",203.94,3.77,17.23
1,PA,972,"217,682.53",223.95,3.85,11.23
2,BA,3371,"615,203.62",182.50,3.86,12.10
3,CE,1332,"278,311.31",208.94,3.86,13.80
4,RJ,12839,"2,142,349.25",166.86,3.88,12.12
5,PE,1648,"324,394.73",196.84,4.01,9.57
6,PB,534,"140,125.06",262.41,4.01,10.49
7,ES,2027,"325,490.67",160.58,4.04,10.61
8,GO,2011,"349,377.13",173.73,4.04,6.57
9,DF,1969,"326,095.81",165.61,4.07,5.74


## Resumen analitico de los estados con mayor porcentaje de pedidos tardíos

El análisis revela que los retrasos logísticos no se distribuyen de forma homogénea entre los distintos estados. Territorios como Maranhão (`MA`), Ceará (`CE`), Río de Janeiro (`RJ`) y Bahía (`BA`) presentan porcentajes de pedidos tardíos claramente superiores a la media, lo que sugiere la existencia de focos geográficos de fricción operativa.

Además, en varios de estos estados los mayores niveles de retraso coinciden con puntuaciones medias de reseñas relativamente más bajas, lo que refuerza la hipótesis de una relación entre desempeño logístico y satisfacción del cliente. Este resultado es especialmente relevante en estados con peso comercial considerable, como `RJ` y `BA`, ya que el impacto potencial de una logística deficiente puede ser mayor tanto en ventas como en experiencia de cliente.

In [937]:
fig = px.scatter_geo(
    state_map,
    lat="lat",
    lon="lon",
    size="pedidos",
    color="porcentaje_tardios",
    hover_name="geolocation_state",
    hover_data={
        "ventas": ":,.0f",
        "pedidos": ":,.0f",
        "review_score_promedio": ":.2f",
        "porcentaje_tardios": ":.2f",
        "lat": False,
        "lon": False
    },
    scope="south america",
    title="Porcentaje de pedidos tardíos por estado",
    size_max=28,
    height=560,
    width=900
)

fig.update_geos(
    center={"lat": -14.2, "lon": -51.9},
    projection_scale=4.0,
    showcountries=True,
    showland=True
)

fig.update_layout(
    title_x=0.5,
    coloraxis_colorbar_title="% tardíos",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()

### Interpretación del porcentaje de pedidos tardíos por estado

El mapa permite identificar de forma visual qué estados presentan una mayor proporción de pedidos tardíos y, al mismo tiempo, cuál es el peso relativo de cada territorio en términos de volumen de pedidos. El color muestra la intensidad del retraso, mientras que el tamaño de la burbuja refleja la dimensión del mercado.

Desde una perspectiva de negocio, este análisis permite priorizar territorios donde la fricción logística tiene un impacto más relevante. No todos los estados con alta tasa de retraso representan el mismo riesgo: aquellos que combinan una proporción elevada de tardíos con un volumen alto de pedidos son los que requieren mayor atención operativa.

In [938]:
top10_sales_categories["ventas_label"] = top10_sales_categories["ventas"].round(0)
top10_sales_categories["ticket_label"] = top10_sales_categories["ticket_promedio"].round(2)
top10_sales_categories["tardios_label"] = top10_sales_categories["porcentaje_tardios"].round(2)
top10_sales_categories["review_label"] = top10_sales_categories["review_score_promedio"].round(2)

## Conclusiones de las diferencias geográficas en retrasos

El análisis de los estados con mayor porcentaje de pedidos tardíos permite detectar si existen zonas geográficas donde la operación logística presenta mayores dificultades. Estas diferencias pueden estar relacionadas con factores como distancia, infraestructura, densidad operativa o complejidad de distribución.

La identificación de estos estados es especialmente relevante porque permite localizar territorialmente los posibles focos de fricción logística y entender mejor cómo la geografía puede influir en la experiencia del cliente.

In [939]:
fig = px.scatter(
    state_summary[state_summary["pedidos"] >= 500],
    x="review_score_promedio",
    y="porcentaje_tardios",
    size="ventas",
    color="geolocation_state",
    hover_name="geolocation_state",
    title="Relación entre satisfacción, retrasos y ventas por estado"
)

fig.update_layout(showlegend=False)
fig.show()

### Resumen de la relación entre satisfacción, retrasos y ventas por estado

Esta visualización integra tres dimensiones clave del negocio: satisfacción del cliente, desempeño logístico y peso comercial del territorio. Los estados situados más a la derecha presentan mejor valoración media, mientras que los ubicados más arriba concentran una mayor proporción de retrasos. El tamaño de cada burbuja añade una lectura de relevancia económica a través del volumen de ventas.

La utilidad de este gráfico radica en que permite priorizar decisiones de forma más estratégica. No todos los problemas operativos tienen el mismo impacto: un estado con retrasos elevados y alto peso comercial representa una prioridad mucho más relevante que otro con problemas similares, pero menor contribución al negocio.

# Analisis de clientes:
- pretende analizar factores relevantes como cuantos clientes compran una vez, cuantos repiten, gastos total por cliente, ticket promedio por cliente y una primera segmentación

In [940]:
customer_summary = (
    master_df
    .groupby("customer_unique_id", as_index=False)
    .agg(
        total_pedidos=("order_id", "count"),
        gasto_total=("payment_value_total", "sum"),
        ticket_promedio=("payment_value_total", "mean"),
        review_score_promedio=("review_score", "mean")
    )
)

customer_summary["gasto_total"] = customer_summary["gasto_total"].round(2)
customer_summary["ticket_promedio"] = customer_summary["ticket_promedio"].round(2)
customer_summary["review_score_promedio"] = customer_summary["review_score_promedio"].round(2)

customer_summary.head()

,customer_unique_id,total_pedidos,gasto_total,ticket_promedio,review_score_promedio
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.00
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.00
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.00
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,4.00
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,5.00


In [941]:
customer_summary["segmento_frecuencia"] = np.where(
    customer_summary["total_pedidos"] == 1,
    "One-time buyer",
    "Repeat buyer"
)

customer_summary["segmento_frecuencia"].value_counts()

segmento_frecuencia
One-time buyer    93099
Repeat buyer       2997
Name: count, dtype: int64

In [942]:
segmento_frecuencia_df = (
    customer_summary["segmento_frecuencia"]
    .value_counts()
    .rename_axis("Segmento")
    .reset_index(name="Clientes")
)

segmento_frecuencia_df["Porcentaje"] = (
    segmento_frecuencia_df["Clientes"] / segmento_frecuencia_df["Clientes"].sum() * 100
).round(2)

segmento_frecuencia_df

,Segmento,Clientes,Porcentaje
0,One-time buyer,93099,96.88
1,Repeat buyer,2997,3.12


## Interpretación de la frecuencia de compra de los clientes

La segmentación inicial muestra una diferencia muy marcada entre clientes de compra única y clientes repetidores. La gran mayoría de los clientes realiza un solo pedido, mientras que solo una fracción reducida vuelve a comprar más de una vez.

Este resultado sugiere que el ecommerce presenta una recurrencia limitada, lo que puede interpretarse como una oportunidad de mejora en términos de fidelización, retención y recompra. En consecuencia, el análisis de clientes no solo permite entender el volumen de compradores, sino también evaluar la capacidad del negocio para convertir compras puntuales en relaciones comerciales más duraderas.

# Resumen por segmento de clientes:

In [943]:
segment_value_summary = (
    customer_summary
    .groupby("segmento_frecuencia", as_index=False)
    .agg(
        clientes=("customer_unique_id", "count"),
        gasto_total_promedio=("gasto_total", "mean"),
        ticket_promedio_cliente=("ticket_promedio", "mean"),
        review_score_promedio=("review_score_promedio", "mean")
    )
)

segment_value_summary["gasto_total_promedio"] = segment_value_summary["gasto_total_promedio"].round(2)
segment_value_summary["ticket_promedio_cliente"] = segment_value_summary["ticket_promedio_cliente"].round(2)
segment_value_summary["review_score_promedio"] = segment_value_summary["review_score_promedio"].round(2)

segment_value_summary

,segmento_frecuencia,clientes,gasto_total_promedio,ticket_promedio_cliente,review_score_promedio
0,One-time buyer,93099,161.82,161.82,4.08
1,Repeat buyer,2997,314.99,148.50,4.12


In [944]:
segment_value_summary["segmento_frecuencia"] = segment_value_summary["segmento_frecuencia"].replace({
    "One-time buyer": "Compra única",
    "Repeat buyer": "Cliente repetidor"
})

segment_value_summary

,segmento_frecuencia,clientes,gasto_total_promedio,ticket_promedio_cliente,review_score_promedio
0,Compra única,93099,161.82,161.82,4.08
1,Cliente repetidor,2997,314.99,148.50,4.12


## Conclusiones del valor por segmento de cliente

La comparación entre clientes de compra única y clientes repetidores muestra que ambos segmentos no aportan el mismo valor al negocio. Aunque la base de clientes está claramente dominada por compradores de una sola vez, los clientes repetidores presentan un gasto total promedio significativamente mayor, lo que confirma su importancia estratégica desde el punto de vista económico.

Sin embargo, esta diferencia no se explica por un ticket promedio más alto en cada pedido, sino por una mayor frecuencia de compra. En otras palabras, los clientes repetidores generan más valor no porque gasten más en cada transacción, sino porque vuelven a comprar. Este hallazgo sugiere que una mejora en la retención y fidelización podría tener un impacto directo en el rendimiento del ecommerce.

In [945]:
segmento_plot = customer_summary["segmento_frecuencia"].value_counts().reset_index().replace({"One-time buyer": "Compra Unica", "Repeat buyer": "Cliente repetidor"})
segmento_plot.columns = ["segmento_frecuencia", "clientes"]

fig = px.pie(
    segmento_plot,
    names="segmento_frecuencia",
    values="clientes",
    hole=0.45,
    title="Distribución de clientes por frecuencia de compra"
)

fig.show()

## Interpretación del gasto total promedio por segmento

Este gráfico permite comparar el gasto acumulado promedio de los clientes según su frecuencia de compra. La comparación es relevante porque ayuda a determinar si los clientes repetidores, aunque menos numerosos, generan un valor económico superior al de los clientes de compra única.

Si la diferencia es significativa, ello refuerza la importancia estratégica de la retención y fidelización, ya que un número pequeño de clientes recurrentes podría aportar un valor desproporcionadamente alto al negocio.

In [946]:
segment_value_summary["segmento_frecuencia"] = segment_value_summary["segmento_frecuencia"].replace({
    "One-time buyer": "Compra única",
    "Repeat buyer": "Cliente repetidor"
})

fig = px.bar(
    segment_value_summary,
    x="segmento_frecuencia",
    y="gasto_total_promedio",
    color="segmento_frecuencia",
    title="Gasto total promedio por segmento de cliente",
    text="gasto_total_promedio"
)

fig.update_layout(
    xaxis_title="Segmento",
    yaxis_title="Gasto total promedio"
)

fig.show()

## Resumen del ticket promedio por segmento

En este gráfico se compara el valor medio de los pedidos entre clientes de compra única y clientes repetidores. Esta métrica permite evaluar si la diferencia entre segmentos no solo depende del número de compras realizadas, sino también del importe medio gastado en cada transacción.

La lectura de este resultado es útil para entender si los clientes repetidores son más valiosos únicamente porque compran más veces o también porque cada pedido individual tiene un mayor valor.

In [947]:
customer_summary["gasto_total"].describe()

count   96,096.00
mean       166.59
std        231.43
min          0.00
25%         63.12
50%        108.00
75%        183.53
max     13,664.08
Name: gasto_total, dtype: float64

In [948]:
fig = px.bar(
    segment_value_summary,
    x="segmento_frecuencia",
    y="ticket_promedio_cliente",
    color="segmento_frecuencia",
    title="Ticket promedio por segmento de cliente",
    text="ticket_promedio_cliente"
)

fig.update_layout(
    xaxis_title="Segmento",
    yaxis_title="Ticket promedio"
)

fig.show()

### Aclaraciones del ticket promedio por segmento de cliente

La comparación del ticket promedio por segmento permite observar el valor medio de cada transacción según el tipo de cliente. En este caso, el análisis muestra que los clientes de compra única presentan un ticket promedio ligeramente superior al de los clientes repetidores.

Desde una perspectiva de negocio, este hallazgo sugiere que el mayor valor de los clientes repetidores no proviene necesariamente de compras unitarias más altas, sino de su recurrencia a lo largo del tiempo. Es decir, aunque cada pedido individual pueda ser similar o incluso algo menor, el cliente repetidor aporta más valor acumulado al negocio por la frecuencia de compra.

Este resultado es importante porque orienta la estrategia comercial hacia dos frentes: por un lado, mantener acciones de captación que impulsen compras iniciales de buen valor, y por otro, reforzar acciones de fidelización que incrementen la recurrencia y el valor de vida del cliente.

### Distribucion del gasto total por cliente

In [949]:
customer_summary["segmento_frecuencia"] = customer_summary["segmento_frecuencia"].replace({
    "One-time buyer": "Compra única",
    "Repeat buyer": "Cliente repetidor"
})

In [950]:
customer_box = customer_summary.copy()
p99 = customer_box["gasto_total"].quantile(0.99)
customer_box = customer_box[customer_box["gasto_total"] <= p99]

fig = px.box(
    customer_box,
    x="segmento_frecuencia",
    y="gasto_total",
    color="segmento_frecuencia",
    points="outliers",
    title="Distribución del gasto total por cliente (hasta percentil 99)"
)

fig.update_layout(
    xaxis_title="Segmento de cliente",
    yaxis_title="Gasto total (R$)",
    title_x=0.5,
    showlegend=False
)

fig.show()

### Conclusiones de la distribución del gasto total por cliente

La distribución del gasto total por cliente permite observar hasta qué punto el valor del ecommerce se concentra en determinados perfiles. Al segmentar entre compra única y cliente repetidor, se evidencia la diferencia entre quienes realizan una sola transacción y quienes acumulan valor a lo largo del tiempo.

Desde una perspectiva de negocio, esta visualización muestra que los clientes repetidores tienden a concentrar un gasto total más elevado, incluso aunque representen una parte menor de la base. Esto refuerza la importancia estratégica de la retención y la recurrencia como palancas de crecimiento sostenible.

# TOP 10 de clientes de mayor valor:

In [951]:
top_customers = (
    customer_summary
    .sort_values("gasto_total", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_customers

,customer_unique_id,total_pedidos,gasto_total,ticket_promedio,review_score_promedio,segmento_frecuencia
0,0a0a92112bd4c708ca5fde585afaa872,1,"13,664.08","13,664.08",1.00,Compra única
1,46450c74a0d8c5ca9395da1daac6c120,3,"9,553.02","3,184.34",1.00,Cliente repetidor
2,da122df9eeddfedc1dc1f5349a1a690c,2,"7,571.63","3,785.82",5.00,Cliente repetidor
3,763c8b1c9c68a0229c42c9fc6f662b93,1,"7,274.88","7,274.88",1.00,Compra única
4,dc4802a71eae9be1dd28f5d788ceb526,1,"6,929.31","6,929.31",5.00,Compra única
5,459bef486812aa25204be022145caa62,1,"6,922.21","6,922.21",NaN,Compra única
6,ff4159b92c40ebe40454e3e6a7c35ed6,1,"6,726.66","6,726.66",5.00,Compra única
7,4007669dec559734d6f53e029e360987,1,"6,081.54","6,081.54",1.00,Compra única
8,5d0a2980b292d049061542014e8960bf,1,"4,809.44","4,809.44",1.00,Compra única
9,eebb5dda148d3893cdaf5b5ca3040ccb,1,"4,764.34","4,764.34",4.00,Compra única


## Interpretación de los clientes de mayor valor

El ranking de clientes con mayor gasto acumulado muestra que el valor económico del ecommerce no se concentra exclusivamente en clientes repetidores. Entre los clientes de mayor valor aparecen varios casos de compra única con importes muy elevados, lo que indica la existencia de pedidos puntuales de alto valor dentro de la base de clientes.

Al mismo tiempo, también se observan clientes repetidores con un gasto acumulado significativo, lo que confirma que la recurrencia contribuye de forma importante al valor total generado. En conjunto, este resultado sugiere que el ecommerce combina dos perfiles relevantes: clientes que realizan compras únicas de importe elevado y clientes que, a través de varias transacciones, construyen un valor acumulado más alto a lo largo del tiempo.